# Factor Extraction: Stability, News, and Panel Size

**Module 04 — Notebook 03**  
**Estimated time:** 15 minutes

## Learning Objectives

1. Measure factor loading stability across an expanding window
2. Decompose a nowcast update into indicator-level news contributions
3. Quantify how panel size (N=2, 3, 4 indicators) affects factor quality and MIDAS RMSE
4. Identify the point at which adding more indicators stops improving the nowcast

## Why This Matters

A factor is only useful if it is stable — if the loadings shift dramatically over time, the extracted factor at step $t$ means something different from the factor at step $t+10$. This notebook diagnoses stability, explains how each new indicator release moves the nowcast (news decomposition), and tests whether a larger panel always helps.

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

In [ ]:
section_divider("1. Data Loading")

In [ ]:
learning_objectives(["Measure factor loading stability across an expanding window", "Decompose a nowcast update into indicator-level news contributions", "Quantify how panel size (N=2, 3, 4 indicators) affects factor quality and MIDAS RMSE", "Identify the point at which adding more indicators stops improving the nowcast"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Data Loading

We use the same GDP and IP CSVs from Module 00, augmented with three synthetic
indicators that mimic payrolls, retail sales, and a financial variable. Each
synthetic indicator shares a common factor with IP but adds idiosyncratic noise.

In [ ]:
def find_csv(filename):
    """Search candidate directories for a CSV file."""
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    for base in candidates:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True).squeeze()
    raise FileNotFoundError(f"Cannot locate {filename}")


gdp_growth = find_csv('gdp_quarterly.csv')
ip_growth  = find_csv('industrial_production_monthly.csv')

ip_vals = ip_growth.values
n_months = len(ip_vals)

# --- Build a 4-indicator monthly panel ---
# Each indicator = signal * IP + idiosyncratic noise
# Signal strength decreases from IP to the financial proxy
rng = np.random.default_rng(99)
payrolls_syn = 0.75 * ip_vals + 0.45 * rng.standard_normal(n_months)
retail_syn   = 0.65 * ip_vals + 0.55 * rng.standard_normal(n_months)
finance_syn  = 0.40 * ip_vals + 0.85 * rng.standard_normal(n_months)

panel_full = pd.DataFrame({
    'IP':       ip_vals,
    'Payrolls': payrolls_syn,
    'Retail':   retail_syn,
    'Finance':  finance_syn,
}, index=ip_growth.index).dropna()

INDICATOR_NAMES = list(panel_full.columns)

print(f"Panel: {panel_full.shape[1]} indicators, {len(panel_full)} monthly observations")
print(f"GDP: {len(gdp_growth)} quarterly observations")
print("\nCorrelation matrix:")
print(panel_full.corr().round(3))

In [ ]:
section_divider("2. Shared Utilities")

## 2. Shared Utilities

The helper functions below are used throughout this notebook. They implement:
- `beta_weights`: Beta polynomial weight vector for K lags
- `estimate_midas`: Profile NLS for Beta MIDAS (returns alpha, beta, theta1, theta2)
- `build_midas_from_series`: Aligns a monthly series with quarterly GDP into a MIDAS matrix
- `extract_factor`: PCA extraction with sign normalization (IP loading positive)

In [ ]:
def beta_weights(K, t1, t2):
    """Beta polynomial weight vector, normalised to sum to 1."""
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, max(t1, 0.01), max(t2, 0.01))
    return raw / raw.sum() if raw.sum() > 1e-12 else np.ones(K) / K


def estimate_midas(Y, X, starts=None):
    """Profile NLS for Beta MIDAS. Returns dict with alpha, beta, theta1, theta2, sse."""
    K = X.shape[1]
    if starts is None:
        starts = [(1.0, 5.0), (1.5, 4.0), (2.0, 3.0), (1.0, 1.0)]

    def psse(theta):
        t1, t2 = theta
        if t1 <= 0.01 or t2 <= 0.01:
            return 1e10
        xw = X @ beta_weights(K, t1, t2)
        xc, yc = xw - xw.mean(), Y - Y.mean()
        ss = xc @ xc
        if ss < 1e-12:
            return float(np.sum((Y - Y.mean()) ** 2))
        b = yc @ xc / ss
        a = Y.mean() - b * xw.mean()
        return float(np.sum((Y - a - b * xw) ** 2))

    results = [
        minimize(psse, s, method='Nelder-Mead',
                 options={'maxiter': 20000, 'xatol': 1e-8, 'fatol': 1e-10})
        for s in starts
    ]
    best = min(results, key=lambda r: r.fun)
    t1, t2 = max(best.x[0], 0.01), max(best.x[1], 0.01)
    w = beta_weights(K, t1, t2)
    xw = X @ w
    xc, yc = xw - xw.mean(), Y - Y.mean()
    b = yc @ xc / (xc @ xc)
    a = float(Y.mean() - b * xw.mean())
    sse = float(np.sum((Y - a - b * xw) ** 2))
    return {'alpha': a, 'beta': float(b), 'theta1': t1, 'theta2': t2, 'weights': w, 'sse': sse}


def build_midas_from_series(y_quarterly, x_monthly, K):
    """
    Align a monthly series with quarterly GDP into a MIDAS (Y, X) pair.
    x_monthly can be any pd.Series with monthly DatetimeIndex or PeriodIndex.
    """
    y_q = y_quarterly.copy()
    y_q.index = y_quarterly.index.to_period('Q')
    x_m = x_monthly.copy()
    x_m.index = x_monthly.index.to_period('M')

    rows_Y, rows_X = [], []
    for q in y_q.index:
        last = q.asfreq('M', how='end')
        lags = [last - i for i in range(K)]
        if any(l not in x_m.index for l in lags):
            continue
        rows_Y.append(y_q[q])
        rows_X.append([x_m[l] for l in lags])
    return np.array(rows_Y), np.array(rows_X)


def extract_factor(X_std, n_factors=1, ref_col=0):
    """
    Extract factors via PCA and apply sign normalisation so ref_col
    (IP by default) has a positive loading on each factor.
    Returns F (T, n_factors), Lambda (N, n_factors), var_exp.
    """
    pca = PCA(n_components=n_factors)
    F = pca.fit_transform(X_std)
    Lambda = pca.components_.T.copy()  # (N, n_factors)

    for j in range(n_factors):
        if Lambda[ref_col, j] < 0:
            F[:, j] = -F[:, j]
            Lambda[:, j] = -Lambda[:, j]

    return F, Lambda, pca.explained_variance_ratio_


print("Helper functions defined.")

In [ ]:
section_divider("3. Factor Loading Stability")

## 3. Factor Loading Stability

At each point in an expanding-window evaluation, we re-estimate the factor model
on training data only. The question is: **do the loadings change substantially as
the training window grows?**

Stable loadings mean the factor has a consistent economic interpretation. Unstable
loadings suggest structural change or that the sample is too small for reliable PCA.

We track the Factor 1 loading for each indicator across expanding windows from
`t_min` (roughly 5 years of monthly data) to the full sample.

In [ ]:
# Standardize the full panel once (for reference)
scaler_full = StandardScaler()
X_full_std = scaler_full.fit_transform(panel_full.values)

N = panel_full.shape[1]
T_monthly = len(panel_full)

# Minimum training window: 60 months (5 years)
T_MIN_MONTHS = 60
step_months = 6  # record every 6 months to keep computation fast

t_grid = list(range(T_MIN_MONTHS, T_monthly, step_months)) + [T_monthly]

# Store loadings at each t
loading_history = {name: [] for name in INDICATOR_NAMES}  # F1 loading per indicator
t_axis = []

for t in t_grid:
    X_sub = panel_full.values[:t]
    scaler_t = StandardScaler()
    X_sub_std = scaler_t.fit_transform(X_sub)

    _, Lambda_t, _ = extract_factor(X_sub_std, n_factors=1, ref_col=0)

    for i, name in enumerate(INDICATOR_NAMES):
        loading_history[name].append(Lambda_t[i, 0])
    t_axis.append(t)

t_axis = np.array(t_axis)

# Compute loading range (max - min) as a stability metric
print("Factor 1 loading stability (max - min across expanding window):")
print("-" * 50)
for name in INDICATOR_NAMES:
    vals = np.array(loading_history[name])
    rng_v = vals.max() - vals.min()
    mean_v = vals.mean()
    print(f"  {name:<12}: mean = {mean_v:+.4f}   range = {rng_v:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = ['steelblue', 'tomato', 'green', 'darkorange']

for ax, name, color in zip(axes.flat, INDICATOR_NAMES, colors):
    vals = np.array(loading_history[name])
    ax.plot(t_axis, vals, color=color, linewidth=2)
    ax.axhline(vals.mean(), color='black', linewidth=1.2, linestyle='--',
               label=f'Mean = {vals.mean():+.3f}')
    ax.fill_between(t_axis,
                    vals.mean() - vals.std(),
                    vals.mean() + vals.std(),
                    alpha=0.15, color=color, label=f'±1 std ({vals.std():.3f})')
    ax.set_title(f'{name} — Factor 1 Loading')
    ax.set_xlabel('Training window size (months)')
    ax.set_ylabel('Loading')
    ax.legend(fontsize=8)
    ax.set_ylim(-0.1, 0.9)

plt.suptitle('Expanding-Window Factor 1 Loading Stability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('loading_stability.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: loading_stability.png")

In [ ]:
section_divider("4. News Decomposition")

**Interpretation:** Loadings converge as the training window grows. Early in the
window the loadings are noisy (PCA estimated on only ~60 observations). By the
time 120+ months of data are available, loadings stabilize. Indicators with higher
signal-to-noise ratio (IP, Payrolls) stabilize faster than noisier series (Finance).

In practice: if loadings are still shifting substantially at the end of the
expanding window, the factor model has structural instability and bootstrap
inference should account for loading uncertainty.

## 4. News Decomposition

The *news* from a data release is how much the nowcast changes when the new
observation arrives. Formally, for a MIDAS model with Beta weights $w_j(\theta)$:

$$\Delta\hat{y}_t = \beta \cdot w_0(\theta) \cdot (x_{mt} - \mathbb{E}[x_{mt}])$$

where $w_0$ is the weight on the most recent lag and $x_{mt} - \mathbb{E}[x_{mt}]$
is the surprise (actual minus expected, approximated by prior mean).

For FA-MIDAS, we first map each indicator surprise through the factor loading,
then through the MIDAS weight:

$$\Delta\hat{y}_t^{\text{FA}} = \beta \cdot w_0(\theta) \cdot \sum_{n=1}^{N} \hat{\lambda}_n \cdot \text{surprise}_n$$

This decomposes the nowcast update into indicator-level contributions.

In [ ]:
# --- Estimate full-sample FA-MIDAS (3-indicator panel) ---
# We use the first 3 indicators (IP + Payrolls + Retail) as in Notebook 02
K = 12
panel_3 = panel_full[['IP', 'Payrolls', 'Retail']]

scaler_3 = StandardScaler()
X3_std = scaler_3.fit_transform(panel_3.values)
F3, Lambda3, _ = extract_factor(X3_std, n_factors=1, ref_col=0)

# Factor series aligned to monthly dates
factor_series = pd.Series(F3[:, 0], index=panel_3.index, name='Factor1')

# Build MIDAS matrix from factor
Y, X_f = build_midas_from_series(gdp_growth, factor_series, K)
est = estimate_midas(Y, X_f)

print(f"Full-sample FA-MIDAS (N=3, K={K}):")
print(f"  alpha  = {est['alpha']:.4f}")
print(f"  beta   = {est['beta']:.4f}")
print(f"  theta1 = {est['theta1']:.4f}")
print(f"  theta2 = {est['theta2']:.4f}")
r2 = 1 - est['sse'] / np.sum((Y - Y.mean()) ** 2)
print(f"  R²     = {r2:.4f}")

In [ ]:
def compute_news_decomposition(beta_hat, weights, Lambda_vec, scaler, panel_df,
                                t_idx, indicator_names):
    """
    Compute the news contribution of each indicator for a given monthly release.

    Parameters
    ----------
    beta_hat : float — MIDAS slope coefficient
    weights  : np.ndarray (K,) — MIDAS weight vector
    Lambda_vec : np.ndarray (N,) — Factor 1 loadings
    scaler   : StandardScaler fitted on the panel
    panel_df : pd.DataFrame — raw monthly panel
    t_idx    : int — month index of the release (0-based)
    indicator_names : list of str

    Returns
    -------
    dict mapping indicator -> news contribution
    """
    # w_0 is the weight on the most recent monthly observation (lag 0)
    w0 = weights[0]

    # Standardized current values
    raw_current = panel_df.values[t_idx]
    std_current = (raw_current - scaler.mean_) / scaler.scale_

    # Standardized prior-month values as a proxy for expected values
    # (In real applications, use model forecasts for the expectation)
    if t_idx == 0:
        raw_prior = panel_df.values[t_idx]
    else:
        raw_prior = panel_df.values[t_idx - 1]
    std_prior = (raw_prior - scaler.mean_) / scaler.scale_

    # Surprise = actual - expected (in standardized units)
    surprise = std_current - std_prior

    # News from each indicator n:
    #   news_n = beta * w0 * lambda_n * surprise_n
    news = {name: float(beta_hat * w0 * Lambda_vec[i] * surprise[i])
            for i, name in enumerate(indicator_names)}
    return news


# Lambda3[:,0] are the Factor 1 loadings for the 3-indicator panel
Lambda3_f1 = Lambda3[:, 0]  # shape (3,)

# Compute news for the last 20 monthly observations
n_recent = 20
t_start = len(panel_3) - n_recent

news_records = []
for t in range(t_start, len(panel_3)):
    nd = compute_news_decomposition(
        est['beta'], est['weights'], Lambda3_f1,
        scaler_3, panel_3, t, ['IP', 'Payrolls', 'Retail']
    )
    nd['date'] = panel_3.index[t]
    news_records.append(nd)

news_df = pd.DataFrame(news_records).set_index('date')
news_df['Total'] = news_df.sum(axis=1)

print("News contributions (last 20 months):")
print(news_df.round(5).to_string())

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: Stacked bar chart of news contributions
t_labels = [str(d)[:7] for d in news_df.index]
x_pos = np.arange(len(news_df))
bar_colors = {'IP': 'steelblue', 'Payrolls': 'tomato', 'Retail': 'green'}

bottom_pos = np.zeros(len(news_df))
bottom_neg = np.zeros(len(news_df))
for col in ['IP', 'Payrolls', 'Retail']:
    vals = news_df[col].values
    pos_vals = np.where(vals >= 0, vals, 0)
    neg_vals = np.where(vals < 0, vals, 0)
    ax_top.bar(x_pos, pos_vals, bottom=bottom_pos, color=bar_colors[col],
               alpha=0.85, label=col)
    ax_top.bar(x_pos, neg_vals, bottom=bottom_neg, color=bar_colors[col],
               alpha=0.85)
    bottom_pos += pos_vals
    bottom_neg += neg_vals

ax_top.axhline(0, color='black', linewidth=0.8)
ax_top.set_ylabel('News contribution to nowcast (pp)')
ax_top.set_title('News Decomposition: Indicator Contributions to Nowcast Update')
ax_top.legend(fontsize=9)

# Bottom: Total news line
ax_bot.plot(x_pos, news_df['Total'].values, color='black', linewidth=2,
            marker='o', markersize=4, label='Total news')
ax_bot.axhline(0, color='gray', linewidth=0.7, alpha=0.6)
ax_bot.set_ylabel('Total nowcast revision (pp)')
ax_bot.set_xlabel('Month')
ax_bot.set_title('Total Nowcast Revision')
ax_bot.set_xticks(x_pos[::2])
ax_bot.set_xticklabels(t_labels[::2], rotation=30, ha='right', fontsize=8)
ax_bot.legend(fontsize=9)

plt.suptitle('News Decomposition: Last 20 Months', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('news_decomposition.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: news_decomposition.png")

In [ ]:
section_divider("5. Panel Size Sensitivity")

**Interpretation:** Each bar represents the contribution of one indicator release
to the nowcast update for that month. IP typically dominates because it has the
highest Factor 1 loading. In months where IP and Payrolls move in opposite
directions, they partially cancel. The total revision (bottom panel) is the net
effect — the amount by which the GDP nowcast would move if you updated the model
with one more month of data.

## 5. Panel Size Sensitivity

More indicators should provide a better-estimated factor, but only if the new
indicators add signal rather than noise. We compare four panel configurations:

| N | Indicators                         |
|---|------------------------------------|
| 1 | IP only (single-indicator MIDAS)   |
| 2 | IP + Payrolls                      |
| 3 | IP + Payrolls + Retail             |
| 4 | IP + Payrolls + Retail + Finance   |

For each configuration we run an expanding-window MIDAS evaluation and record RMSE.

In [ ]:
def expanding_window_rmse(Y_quarterly, monthly_panel, K, min_train_quarters,
                           n_factors=1, use_raw_ip=False):
    """
    Expanding-window RMSE for FA-MIDAS (or single-indicator MIDAS if use_raw_ip=True).

    Parameters
    ----------
    Y_quarterly     : pd.Series — quarterly GDP growth
    monthly_panel   : pd.DataFrame — balanced monthly indicator panel
    K               : int — MIDAS lag order
    min_train_quarters : int — minimum training periods (in quarters)
    n_factors       : int — number of factors to extract
    use_raw_ip      : bool — if True, skip PCA and use IP column directly

    Returns
    -------
    rmse : float
    n_valid : int
    """
    # Pre-align Y with a full-sample MIDAS index to get T_q
    y_q = Y_quarterly.copy()
    y_q.index = Y_quarterly.index.to_period('Q')
    T_q = len(y_q)
    quarters = list(y_q.index)

    sq_errors = []

    for t in range(min_train_quarters, T_q):
        # --- Training data: quarters 0..t-1 ---
        # Find the last month available at end of quarter t-1
        last_train_q = quarters[t - 1]
        last_train_m = last_train_q.asfreq('M', how='end')
        last_train_date = last_train_m.to_timestamp(how='end')

        panel_train = monthly_panel[monthly_panel.index <= last_train_date]
        if len(panel_train) < K + 12:
            sq_errors.append(np.nan)
            continue

        # --- Extract or use raw series ---
        if use_raw_ip:
            factor_train = panel_train['IP']
        else:
            scaler_t = StandardScaler()
            X_t_std = scaler_t.fit_transform(panel_train.values)
            F_t, Lambda_t, _ = extract_factor(X_t_std, n_factors=n_factors, ref_col=0)
            factor_train = pd.Series(F_t[:, 0], index=panel_train.index)

        # --- Build MIDAS matrix on training quarters ---
        Y_tr_pd = y_q.iloc[:t]
        Y_arr, X_arr = build_midas_from_series(Y_tr_pd.rename(lambda x: x.to_timestamp()),
                                                factor_train, K)
        if len(Y_arr) < 10:
            sq_errors.append(np.nan)
            continue

        est_t = estimate_midas(Y_arr, X_arr)

        # --- Nowcast for quarter t ---
        test_q = quarters[t]
        last_test_m = test_q.asfreq('M', how='end')

        # Get test-quarter factor lags (project onto training factor space)
        if use_raw_ip:
            # Raw IP: just look up values
            ip_m = panel_train.index.to_period('M') if hasattr(panel_train.index, 'to_period') \
                    else pd.PeriodIndex(panel_train.index, freq='M')
            # Extend to full monthly panel for test lags
            ip_full_m = monthly_panel['IP'].copy()
            ip_full_m.index = monthly_panel.index.to_period('M')
            lags = [last_test_m - i for i in range(K)]
            if any(l not in ip_full_m.index for l in lags):
                sq_errors.append(np.nan)
                continue
            x_test = np.array([ip_full_m[l] for l in lags])
        else:
            # FA-MIDAS: project test months onto training PCA space
            full_m_index = monthly_panel.index.to_period('M')
            lags = [last_test_m - i for i in range(K)]
            if any(l not in full_m_index for l in lags):
                sq_errors.append(np.nan)
                continue

            # Get raw test lags from full panel
            rows = []
            full_m_panel = monthly_panel.copy()
            full_m_panel.index = full_m_index
            for lag in lags:
                rows.append(full_m_panel.loc[lag].values)
            X_test_raw = np.array(rows)  # (K, N)

            # Standardize with training scaler
            X_test_std = (X_test_raw - scaler_t.mean_) / scaler_t.scale_

            # Project onto Factor 1 axis (Lambda_t[:,0])
            factor_vec = Lambda_t[:, 0]
            x_test = X_test_std @ factor_vec  # (K,)

            # Apply sign normalization consistent with training extraction
            if Lambda_t[0, 0] < 0:
                x_test = -x_test

        # Nowcast
        w = beta_weights(K, est_t['theta1'], est_t['theta2'])
        xw_test = float(x_test @ w)
        y_hat = est_t['alpha'] + est_t['beta'] * xw_test
        actual = float(y_q.iloc[t])
        sq_errors.append((actual - y_hat) ** 2)

    valid = [e for e in sq_errors if not np.isnan(e)]
    rmse = np.sqrt(np.mean(valid)) if valid else np.nan
    return rmse, len(valid)


print("Expanding-window function defined.")

In [ ]:
# Run panel size comparison
# N=1: raw IP (single-indicator MIDAS, no factor extraction)
# N=2,3,4: FA-MIDAS with increasing panel size
K_eval = 9
MIN_Q = 25  # minimum training quarters

panel_configs = [
    (1, ['IP'],                       True,  'N=1 (IP only)'),
    (2, ['IP', 'Payrolls'],           False, 'N=2 (IP+Payrolls)'),
    (3, ['IP', 'Payrolls', 'Retail'], False, 'N=3 (IP+Payrolls+Retail)'),
    (4, INDICATOR_NAMES,              False, 'N=4 (All indicators)'),
]

rmse_results = []
print(f"Evaluating panel size sensitivity (K={K_eval}, min_train={MIN_Q} quarters)...")
print("(This may take 1-2 minutes)")

for n_ind, cols, use_raw, label in panel_configs:
    panel_sub = panel_full[cols]
    rmse_n, n_valid = expanding_window_rmse(
        gdp_growth, panel_sub, K_eval, MIN_Q,
        n_factors=1, use_raw_ip=use_raw
    )
    rmse_results.append((n_ind, label, rmse_n, n_valid))
    print(f"  {label:<30}: RMSE = {rmse_n:.4f}  (n_valid={n_valid})")

In [ ]:
# AR(1) benchmark
y_q_arr = gdp_growth.values
T_q_total = len(y_q_arr)
ar_sq = []
for t in range(MIN_Q, T_q_total):
    Y_tr = y_q_arr[:t]
    Z = np.column_stack([np.ones(t - 1), Y_tr[:-1]])
    coef, _, _, _ = np.linalg.lstsq(Z, Y_tr[1:], rcond=None)
    ar_sq.append((y_q_arr[t] - (coef[0] + coef[1] * y_q_arr[t - 1])) ** 2)
ar_rmse = float(np.sqrt(np.mean(ar_sq)))
print(f"AR(1) benchmark RMSE: {ar_rmse:.4f}")

In [ ]:
# Summary table
print("\nPanel Size Sensitivity — RMSE Summary")
print("=" * 55)
print(f"{'Model':<35}{'RMSE':>8}{'vs AR(1)':>10}")
print("-" * 55)
print(f"{'AR(1) benchmark':<35}{ar_rmse:>8.4f}{'—':>10}")
for n_ind, label, rmse_n, n_valid in rmse_results:
    delta = (rmse_n - ar_rmse) / ar_rmse * 100
    direction = f"{abs(delta):.1f}% {'better' if delta < 0 else 'worse'}"
    print(f"  {label:<33}{rmse_n:>8.4f}{direction:>10}")
print("=" * 55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: RMSE bar chart
ax_left = axes[0]
n_vals   = [r[0] for r in rmse_results]
rmse_vals = [r[2] for r in rmse_results]
labels_short = ['N=1', 'N=2', 'N=3', 'N=4']
bar_cols = ['#aec7e8', '#1f77b4', '#2ca02c', '#ff7f0e']

bars = ax_left.bar(labels_short, rmse_vals, color=bar_cols, alpha=0.9)
ax_left.axhline(ar_rmse, color='tomato', linewidth=2, linestyle='--',
                label=f'AR(1): {ar_rmse:.4f}')
for bar, rmse_v in zip(bars, rmse_vals):
    ax_left.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002,
                 f'{rmse_v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax_left.set_ylabel('OOS RMSE')
ax_left.set_title('RMSE by Panel Size')
ax_left.legend(fontsize=9)
ax_left.set_ylim(0, max(rmse_vals + [ar_rmse]) * 1.3)

# Right: Marginal improvement from adding each indicator
ax_right = axes[1]
marginal = [rmse_vals[0] - rmse_vals[0]]  # baseline = 0
for i in range(1, len(rmse_vals)):
    marginal.append(rmse_vals[i - 1] - rmse_vals[i])

colors_marginal = ['steelblue' if m >= 0 else 'tomato' for m in marginal]
ax_right.bar(labels_short, marginal, color=colors_marginal, alpha=0.85)
ax_right.axhline(0, color='black', linewidth=0.8)
ax_right.set_ylabel('RMSE reduction vs. previous N (pp)')
ax_right.set_title('Marginal RMSE Improvement from Adding Each Indicator')
for i, (label, marg) in enumerate(zip(labels_short, marginal)):
    ax_right.text(i, marg + (0.0005 if marg >= 0 else -0.0015),
                  f'{marg:+.4f}', ha='center', fontsize=9)

plt.suptitle('Panel Size Sensitivity: Single-Indicator vs FA-MIDAS',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('panel_size_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: panel_size_sensitivity.png")

**Interpretation of the marginal improvement chart:**

- Moving from N=1 to N=2 (adding Payrolls) typically yields the largest gain because
  Payrolls carries substantial business-cycle signal orthogonal to IP.
- Moving from N=3 to N=4 (adding Finance) often provides smaller or no gain if
  Finance adds mostly idiosyncratic noise relative to its signal content.
- When the marginal bar turns negative (adding an indicator *increases* RMSE), the
  new series introduces more estimation noise than signal.

This is why Bai-Ng factor count selection matters: it prevents over-fitting by
limiting the factor dimension to genuinely common variation.

## 6. Combined Diagnostic Summary

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- Panel A: Loading stability (IP and Payrolls) ---
ax_A = fig.add_subplot(gs[0, 0])
for name, color in [('IP', 'steelblue'), ('Payrolls', 'tomato')]:
    vals = np.array(loading_history[name])
    ax_A.plot(t_axis, vals, color=color, linewidth=2, label=name)
ax_A.set_xlabel('Training window (months)')
ax_A.set_ylabel('F1 Loading')
ax_A.set_title('Loading Stability')
ax_A.legend(fontsize=8)

# --- Panel B: Loading stability (Retail and Finance) ---
ax_B = fig.add_subplot(gs[0, 1])
for name, color in [('Retail', 'green'), ('Finance', 'darkorange')]:
    vals = np.array(loading_history[name])
    ax_B.plot(t_axis, vals, color=color, linewidth=2, label=name)
ax_B.set_xlabel('Training window (months)')
ax_B.set_ylabel('F1 Loading')
ax_B.set_title('Loading Stability (Noisy Indicators)')
ax_B.legend(fontsize=8)

# --- Panel C: News total per month (bar) ---
ax_C = fig.add_subplot(gs[0, 2])
total_news = news_df['Total'].values
bar_c = ['steelblue' if v >= 0 else 'tomato' for v in total_news]
ax_C.bar(np.arange(len(total_news)), total_news, color=bar_c, alpha=0.85)
ax_C.axhline(0, color='black', linewidth=0.8)
ax_C.set_xlabel('Recent months (t-19 to t)')
ax_C.set_ylabel('Total news (pp)')
ax_C.set_title('Total Nowcast News')

# --- Panel D: RMSE by panel size ---
ax_D = fig.add_subplot(gs[1, 0:2])
x_d = np.arange(len(rmse_results) + 1)
labels_d = ['AR(1)'] + [r[1].split('(')[1].rstrip(')') for r in rmse_results]
vals_d = [ar_rmse] + rmse_vals
cols_d = ['#d62728'] + bar_cols
bars_d = ax_D.bar(x_d, vals_d, color=cols_d, alpha=0.9)
for bar, v in zip(bars_d, vals_d):
    ax_D.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
              f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
ax_D.set_xticks(x_d)
ax_D.set_xticklabels(labels_d, fontsize=9)
ax_D.set_ylabel('OOS RMSE')
ax_D.set_title('RMSE: AR(1) vs MIDAS vs FA-MIDAS (N=2,3,4)')
ax_D.set_ylim(0, max(vals_d) * 1.35)

# --- Panel E: Marginal improvement ---
ax_E = fig.add_subplot(gs[1, 2])
marg_x = np.arange(1, len(rmse_vals))
marg_y = [rmse_vals[i - 1] - rmse_vals[i] for i in range(1, len(rmse_vals))]
marg_c = ['steelblue' if m >= 0 else 'tomato' for m in marg_y]
ax_E.bar(marg_x, marg_y, color=marg_c, alpha=0.85)
ax_E.axhline(0, color='black', linewidth=0.8)
ax_E.set_xticks(marg_x)
ax_E.set_xticklabels([f'N={i} → N={i+1}' for i in range(1, len(rmse_vals))], fontsize=8)
ax_E.set_ylabel('RMSE reduction (pp)')
ax_E.set_title('Marginal Value of Each Indicator')

fig.suptitle('Factor Extraction: Stability, News, and Panel Size Summary',
             fontsize=14, fontweight='bold')
plt.savefig('factor_extraction_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved: factor_extraction_summary.png")

## 7. Self-Check

In [ ]:
print("Self-check...")
passed = 0
total = 5

# --- Check 1: Loading history has correct length ---
expected_steps = len(t_axis)
for name in INDICATOR_NAMES:
    assert len(loading_history[name]) == expected_steps, \
        f"FAIL 1: {name} has {len(loading_history[name])} entries, expected {expected_steps}"
passed += 1
print(f"  PASS 1: Loading history has {expected_steps} expanding-window steps")

# --- Check 2: IP loading is always positive (sign normalization holds) ---
ip_loadings = np.array(loading_history['IP'])
assert all(l > 0 for l in ip_loadings), \
    f"FAIL 2: Some IP loadings are non-positive: min={ip_loadings.min():.4f}"
passed += 1
print(f"  PASS 2: IP loading is positive at all {expected_steps} steps (sign normalization correct)")

# --- Check 3: News contributions sum to total ---
computed_total = (news_df['IP'] + news_df['Payrolls'] + news_df['Retail']).values
recorded_total = news_df['Total'].values
max_diff = np.max(np.abs(computed_total - recorded_total))
assert max_diff < 1e-10, \
    f"FAIL 3: News columns do not sum to Total (max diff = {max_diff:.2e})"
passed += 1
print(f"  PASS 3: News columns sum to Total (max diff = {max_diff:.2e})")

# --- Check 4: Panel size RMSE is non-increasing from N=1 to N=3 ---
# (Not guaranteed in general, but with this synthetic data it should hold for N=1..3)
rmse_n1 = rmse_results[0][2]
rmse_n2 = rmse_results[1][2]
rmse_n3 = rmse_results[2][2]
# Allow small tolerance: adding a correlated indicator should not hurt by more than 20%
assert rmse_n2 <= rmse_n1 * 1.2, \
    f"FAIL 4: N=2 RMSE ({rmse_n2:.4f}) is much worse than N=1 ({rmse_n1:.4f})"
passed += 1
print(f"  PASS 4: N=2 RMSE ({rmse_n2:.4f}) within 20% of N=1 ({rmse_n1:.4f})")

# --- Check 5: All MIDAS models beat AR(1) or are close (within 30%) ---
for n_ind, label, rmse_n, n_valid in rmse_results:
    assert rmse_n <= ar_rmse * 1.30, \
        f"FAIL 5: {label} RMSE ({rmse_n:.4f}) is >30% worse than AR(1) ({ar_rmse:.4f})"
passed += 1
print(f"  PASS 5: All models within 30% of AR(1) benchmark")

print(f"\n{'='*45}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Topic | Key Finding |
|-------|-------------|
| Loading stability | Loadings converge after ~120 months; IP and Payrolls are more stable than Finance |
| News decomposition | IP dominates revisions due to highest Factor 1 loading |
| Panel size N=1→2 | Adding a strongly correlated indicator (Payrolls) typically reduces RMSE |
| Panel size N=3→4 | Adding a noisy indicator (Finance) may not improve and can hurt RMSE |
| Practical rule | Select N based on Bai-Ng, not by adding all available series |

### Connections

- **Notebook 01:** PCA factor extraction and Bai-Ng criterion
- **Notebook 02:** FA-MIDAS estimation and expanding-window RMSE
- **Exercises:** `exercises/01_dfm_self_check.py` for hands-on practice
- **Module 03:** Ragged-edge MIDAS and nowcast vintages

### Next Steps

You have now completed Module 04. To consolidate your understanding:

1. Run `exercises/01_dfm_self_check.py` and ensure all 5 exercises pass.
2. Replace the synthetic Payrolls and Retail series with real FRED data (PAYEMS, RSAFS).
3. Extend the news decomposition to a real nowcast calendar — compute daily updates
   as each indicator releases during the quarter.

In [ ]:
key_takeaways(["- Moving from N=1 to N=2 (adding Payrolls) typically yields the largest gain bec", "Moving from N=3 to N=4 (adding Finance) often provides smaller or no gain if", "When the marginal bar turns negative (adding an indicator *increases* RMSE), the"])